In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from torch.optim import AdamW

from tqdm.auto import tqdm
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
print(torch.cuda.is_available())

True


In [ ]:
torch.cuda.get_device_name(0)

'Tesla T4'

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATA_DIR = Path(
    "/content/drive/MyDrive/DigikalaProject/processed"
)


train_df = pd.read_parquet(
    DATA_DIR / "train_group_split.parquet"
)


test_df = pd.read_parquet(
    DATA_DIR / "test_group_split.parquet"
)

In [ ]:
print(train_df.shape)
print(test_df.shape)

(4203084, 8)
(1058769, 8)


In [ ]:
train_sample, _ = train_test_split(
    train_df,
    train_size=150000,
    stratify=train_df["recommendation_status"],
    random_state=42
)


print(train_sample.shape)

(150000, 8)


In [ ]:
train_part, val_part = train_test_split(
    train_sample,
    test_size=0.2,
    stratify=train_sample["recommendation_status"],
    random_state=42
)


print(train_part.shape)
print(val_part.shape)

(120000, 8)
(30000, 8)


In [ ]:
X_train = train_part["text"].astype(str)

y_train = train_part["recommendation_status"]


X_val = val_part["text"].astype(str)

y_val = val_part["recommendation_status"]


X_test = test_df["text"].astype(str)

y_test = test_df["recommendation_status"]

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(
    train_df["recommendation_status"]
)
y_train_encoded = label_encoder.transform(
    y_train
)


y_val_encoded = label_encoder.transform(
    y_val
)


y_test_encoded = label_encoder.transform(
    y_test
)

In [ ]:
from transformers import AutoTokenizer

In [ ]:
MODEL_NAME = "HooshvareLab/bert-base-parsbert-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

In [ ]:
sample_text = "این دوربین کیفیت  و زوم خیلی خوبی دارد"


encoded = tokenizer(
    sample_text
)


encoded

{'input_ids': [2, 2042, 4528, 3564, 331, 19893, 3062, 3090, 2192, 4], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
MAX_LENGTH = 128

In [ ]:
sample = tokenizer(
    X_train.iloc[:3].tolist(),
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True
)

sample.keys()

KeysView({'input_ids': [[2, 3660, 3734, 2031, 4810, 72912, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 3701, 4643, 2320, 73449, 2031, 2339, 5719, 5159, 2968, 3446, 3701, 4643, 16437, 24, 8, 3482, 10097, 2083, 23863, 8, 26, 24, 8, 3701, 4643, 10097, 2986, 8, 26, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 20830, 45362, 3701, 4643, 1176, 3326, 2083, 6575, 331, 57266, 1155, 3062, 25557, 2078, 2073, 5218, 12411, 2031, 

In [ ]:
print(
    len(sample["input_ids"])
)

print(
    len(sample["input_ids"][0])
)

3
128


In [ ]:
from torch.utils.data import Dataset

In [ ]:
class CommentDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length
    ):

        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length


    def __len__(self):

        return len(self.texts)


    def __getitem__(self, idx):

        text = self.texts.iloc[idx]

        label = self.labels[idx]


        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )


        return {

            "input_ids":
            encoding["input_ids"].squeeze(0),


            "attention_mask":
            encoding["attention_mask"].squeeze(0),


            "labels":
            torch.tensor(
                label,
                dtype=torch.long
            )
        }

In [ ]:
MAX_LENGTH = 128

train_dataset = CommentDataset(
    X_train.reset_index(drop=True),
    y_train_encoded,
    tokenizer,
    MAX_LENGTH
)


val_dataset = CommentDataset(
    X_val.reset_index(drop=True),
    y_val_encoded,
    tokenizer,
    MAX_LENGTH
)


test_dataset = CommentDataset(
    X_test.reset_index(drop=True),
    y_test_encoded,
    tokenizer,
    MAX_LENGTH
)

In [ ]:
X_train[0]

'پیشنهاد نمیشود به درد نمیخوره   '

In [ ]:
sample = train_dataset[0]

sample

{'input_ids': tensor([    2,  3660,  3734,  2031,  4810, 72912,     4,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,   

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
batch = next(iter(train_loader))

batch["input_ids"].shape

torch.Size([16, 128])

In [ ]:
from transformers import AutoModelForSequenceClassification

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: HooshvareLab/bert-base-parsbert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(100000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [ ]:
batch = next(iter(train_loader))


input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)


outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels
)


print(outputs.loss)
print(outputs.logits.shape)

tensor(1.2183, device='cuda:0', grad_fn=<NllLossBackward0>)
torch.Size([16, 3])


In [ ]:
from torch.optim import AdamW

In [ ]:
optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

In [ ]:
len(train_dataset)


150000

In [ ]:
def evaluate(model, loader):

    model.eval()

    predictions = []
    labels_all = []


    with torch.no_grad():

        for batch in tqdm(loader):

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)


            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )


            preds = torch.argmax(
                outputs.logits,
                dim=1
            )


            predictions.extend(
                preds.cpu().numpy()
            )

            labels_all.extend(
                labels.cpu().numpy()
            )


    f1 = f1_score(
        labels_all,
        predictions,
        average="macro"
    )


    return f1, labels_all, predictions

In [ ]:
EPOCHS = 2


best_f1 = 0


for epoch in range(EPOCHS):

    print(
        f"\nEpoch {epoch+1}/{EPOCHS}"
    )


    model.train()

    total_loss = 0


    loop = tqdm(train_loader)


    for batch in loop:


        optimizer.zero_grad()


        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)



        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )


        loss = outputs.loss


        loss.backward()


        optimizer.step()


        total_loss += loss.item()


        loop.set_description(
            f"loss {loss.item():.4f}"
        )


    print(
        "Train loss:",
        total_loss / len(train_loader)
    )


    val_f1, _, _ = evaluate(
        model,
        val_loader
    )


    print(
        "Validation Macro F1:",
        val_f1
    )


    if val_f1 > best_f1:

        best_f1 = val_f1


        torch.save(
            model.state_dict(),
            "best_parsbert_model.pt"
        )


        print("Best model saved")


Epoch 1/2


  0%|          | 0/15000 [00:00<?, ?it/s]

Train loss: 0.33894095490043985


  0%|          | 0/3750 [00:00<?, ?it/s]

Validation Macro F1: 0.6275247676686782
Best model saved

Epoch 2/2


  0%|          | 0/15000 [00:00<?, ?it/s]

Train loss: 0.30546029219931614


  0%|          | 0/3750 [00:00<?, ?it/s]

Validation Macro F1: 0.7131004650395504
Best model saved


In [1]:
import os

print(os.path.exists("best_parsbert_model.pt"))

False


In [ ]:
model.load_state_dict(
    torch.load(
        "best_parsbert_model.pt"
    )
)

In [ ]:
test_f1, y_true, y_pred = evaluate(
    model,
    test_loader
)


print(
    "Test Macro F1:",
    test_f1
)

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_
    )
)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)


cm

In [ ]:
plt.figure(figsize=(6,5))

plt.imshow(cm)

plt.xticks(
    range(3),
    label_encoder.classes_,
    rotation=45
)

plt.yticks(
    range(3),
    label_encoder.classes_
)


plt.xlabel("Predicted")

plt.ylabel("True")

plt.title("Confusion Matrix")

plt.colorbar()

plt.show()